In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from yeti_iga.future.bspline import (BSpline, BSplineSurface, ControlPointManager,
    Patch, HRefiner, SubdivisionRefiner, PRefiner, BezierExtractor
)

---
## Bézier extraction

Bézier extraction (Borden et al. 2011) decomposes a B-spline patch into independent
Bernstein elements.  For each element $e$, the extraction operator $C^e$ satisfies:

$$\mathbf{N}^e_\text{active}(\xi) = C^e \cdot \mathbf{B}^p(\hat{\xi})$$

where $\mathbf{B}^p$ is the tensor-product Bernstein basis on $[0,1]^n$ and
$\hat{\xi}_d = (\xi_d - \xi^e_{d,a}) / (\xi^e_{d,b} - \xi^e_{d,a})$ are the
element-local normalised coordinates ($\hat{\xi}_d \in [0,1]$).

The **Bézier control points** of element $e$ are then:
$$P^e_\text{Bézier} = (C^e)^\top P^e_\text{active}$$

In [ ]:
# Demo patch: quadratic in u (p=2), linear in v (p=1)
# 2 elements in u (knot inserted at u=0.5), 1 element in v
mgr_bz = ControlPointManager(dim=2)
for iu in range(4):   # iv=0: bottom row y=0
    mgr_bz.add_point([float(iu), 0.0])
for iu in range(4):   # iv=1: top row y=1
    mgr_bz.add_point([float(iu), 1.0])

kv_u = np.array([0., 0., 0., 0.5, 1., 1., 1.])  # p=2, 4 CPs, 2 elements
kv_v = np.array([0., 0., 1., 1.])                # p=1, 2 CPs, 1 element
su_bz = BSpline(2, kv_u)
sv_bz = BSpline(1, kv_v)
surf_bz = BSplineSurface(su_bz, sv_bz)

nu_bz, nv_bz = 4, 2
patch_bz = Patch(surf_bz, mgr_bz, list(range(nu_bz * nv_bz)), [nu_bz, nv_bz])

# Run Bézier extraction
elems = BezierExtractor.extract_nd(patch_bz)
print(f"Patch: {nu_bz}×{nv_bz} CPs, degrees (pu={su_bz.degree}, pv={sv_bz.degree})")
print(f"Number of elements: {len(elems)}\n")

for elem in elems:
    P_active = np.array([patch_bz.control_point(j) for j in elem.active_indices])
    P_bz = elem.C.T @ P_active
    print(f"Element {tuple(elem.elem_index)}  |  active indices: {list(elem.active_indices)}")
    print(f"  Extraction operator C^e:\n{np.round(elem.C, 3)}")
    print(f"  Bézier CPs (u-fastest):\n{np.round(P_bz, 4)}\n")

In [ ]:
def plot_bezier_elements_2d(patch, title='Bézier extraction'):
    """Plot Bézier control polygons (one colour per element) and the C^e operator."""
    elems = BezierExtractor.extract_nd(patch)
    su = patch.tensor.components[0]
    sv = patch.tensor.components[1]
    pu, pv = su.degree, sv.degree
    u_breaks = np.unique(su.knot_vector)
    v_breaks = np.unique(sv.knot_vector)
    n_elems = len(elems)
    colors = plt.cm.tab10(np.arange(n_elems) % 10)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    # Left: iso-parametric lines + Bézier control polygons
    ax1.set_title(f'{title}  ({n_elems} element(s), degree {pu}×{pv})')
    n_samp = 80
    for u_val in u_breaks:
        vs = np.linspace(v_breaks[0], v_breaks[-1], n_samp)
        spans = np.array([[su.find_span(u_val), sv.find_span(v)] for v in vs], dtype=np.int32)
        pts = patch.evaluate_patch_nd_omp(spans,
                np.column_stack([np.full(n_samp, u_val), vs]))
        ax1.plot(pts[:, 0], pts[:, 1], 'b-', lw=1.2)
    for v_val in v_breaks:
        us = np.linspace(u_breaks[0], u_breaks[-1], n_samp)
        spans = np.array([[su.find_span(u), sv.find_span(v_val)] for u in us], dtype=np.int32)
        pts = patch.evaluate_patch_nd_omp(spans,
                np.column_stack([us, np.full(n_samp, v_val)]))
        ax1.plot(pts[:, 0], pts[:, 1], 'b-', lw=1.2)

    for elem, color in zip(elems, colors):
        P_active = np.array([patch.control_point(j) for j in elem.active_indices])
        P_bz = elem.C.T @ P_active                       # (n_local, dim_phys)
        P_grid = P_bz.reshape(pv + 1, pu + 1, -1)     # (nv_loc, nu_loc, dim)
        for iv in range(pv + 1):
            ax1.plot(P_grid[iv, :, 0], P_grid[iv, :, 1], '--', color=color, lw=1.0)
        for iu in range(pu + 1):
            ax1.plot(P_grid[:, iu, 0], P_grid[:, iu, 1], '--', color=color, lw=1.0)
        ax1.scatter(P_bz[:, 0], P_bz[:, 1], s=40, c=[color], zorder=3)
        ax1.annotate(f'e{tuple(elem.elem_index)}', P_bz.mean(axis=0),
                     ha='center', va='center', fontsize=9,
                     color=color, fontweight='bold')

    ax1.set_aspect('equal')
    ax1.grid(True)

    # Right: extraction operator C^e of the first element
    ax2.set_title(f'Extraction operator $C^e$  (elem {tuple(elems[0].elem_index)})')
    C = elems[0].C
    im = ax2.imshow(C, cmap='Blues', vmin=0, vmax=1)
    plt.colorbar(im, ax=ax2, shrink=0.8)
    for ii in range(C.shape[0]):
        for jj in range(C.shape[1]):
            v = C[ii, jj]
            if v > 1e-10:
                ax2.text(jj, ii, f'{v:.2f}', ha='center', va='center',
                         fontsize=9, color='white' if v > 0.6 else 'black')
    ax2.set_xlabel('Bernstein index $j$', fontsize=10)
    ax2.set_ylabel('B-spline index $i$', fontsize=10)
    plt.tight_layout()
    plt.show()

plot_bezier_elements_2d(patch_bz)

---
## VTU export (Paraview)

`write_bezier_patch_vtu` writes each B-spline element as a high-order VTK Bézier cell:
- 2D patch → **`VTK_BEZIER_QUADRILATERAL`** (cell type 77)
- 3D patch → **`VTK_BEZIER_HEXAHEDRON`** (cell type 79)

Bézier control points per element are computed as $P^e_\text{Bézier} = (C^e)^\top P^e_\text{active}$.
A scalar or vector field defined at the B-spline CPs is transformed the same way,
so Paraview interpolates it correctly inside each curved element.

> **Requirement**: Paraview 5.9+ / VTK 9.0+ for correct high-order Bézier rendering.

In [ ]:
from yeti_iga.future.vtu import write_bezier_patch_vtu
import os

os.makedirs('output', exist_ok=True)

# Geometry only
write_bezier_patch_vtu(patch_bz, 'output/patch_bz.vtu')
print("Geometry exported → output/patch_bz.vtu")

# With a scalar field: distance from each CP to the geometric centre
all_cp = np.array([patch_bz.control_point(i) for i in range(patch_bz.n_cp)])
center = all_cp.mean(axis=0)
field_dist = np.linalg.norm(all_cp - center, axis=1)

write_bezier_patch_vtu(patch_bz, 'output/patch_bz_field.vtu',
                        field=field_dist, field_name='dist_to_centre')
print("With scalar field   → output/patch_bz_field.vtu")

print("\nOpen in Paraview:")
print("  File > Open > output/patch_bz_field.vtu")
print("  Apply  →  patch rendered with curved Bézier edges")
print("  Colour by 'dist_to_centre'")